In [ ]:
import os
import random
import time
import h5py
import numpy as np
import pandas as pd
import scipy.io
import librosa
import kagglehub
import h5py
import IPython.display as ipd
from IPython.display import display, Audio
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score 


from sktime.classification.kernel_based import RocketClassifier
from sktime.transformations.panel.rocket import Rocket


In [ ]:


# ==================================================
# 2. CARGA Y CONCATENACIÓN DE BABBLE NOISE (LOCAL)
# ==================================================
# Definir la ruta local donde ya tenés descargados los audios
babble_exact_path = "datasets/noise/0dB/valid/real" # Ajustá esto al nombre exacto de tu subcarpeta si es necesario

print(f"Cargando audios directamente desde: {babble_exact_path} ...")

audio_buffers_list = []
max_archivos_a_combinar = 40 
sr_original = 16000
sr_objetivo = 22050

archivos_mat = 0
archivos_wav = 0

# Iteramos directamente sobre los archivos de esa única carpeta
for f in os.listdir(babble_exact_path):
    if len(audio_buffers_list) >= max_archivos_a_combinar:
        break  # Salida limpia si ya llegamos al límite

    file_path = os.path.join(babble_exact_path, f)
    
    # Nos aseguramos de procesar solo archivos (ignorando si hay alguna subcarpeta colada)
    if not os.path.isfile(file_path):
        continue

    ext = file_path.lower()
    
    if ext.endswith('.mat'):
        try:
            # Intento A: Formato MAT clásico (SciPy)
            mat_contents = scipy.io.loadmat(file_path)
            keys = [k for k in mat_contents.keys() if not k.startswith('__')]
            if keys:
                audio_raw = mat_contents[keys[0]].flatten().astype(np.float32)
                if audio_raw.size > 0:
                    audio_buffers_list.append(audio_raw)
                    archivos_mat += 1
        except Exception:
            try:
                # Intento B: Formato MAT nuevo (HDF5)
                with h5py.File(file_path, 'r') as f_h5:
                    keys = list(f_h5.keys())
                    if keys:
                        audio_raw = np.array(f_h5[keys[0]]).flatten().astype(np.float32)
                        if audio_raw.size > 0:
                            audio_buffers_list.append(audio_raw)
                            archivos_mat += 1
            except Exception:
                pass  # Archivo corrupto o ilegible
                
    elif ext.endswith('.wav'):
        try:
            y, _ = librosa.load(file_path, sr=sr_original)
            if y.size > 0:
                audio_buffers_list.append(y)
                archivos_wav += 1
        except Exception:
            pass

# Procesamiento final
print("-" * 50)
print("Estadísticas de carga directa:")
print(f" - Archivos .mat procesados con éxito: {archivos_mat}")
print(f" - Archivos .wav procesados con éxito: {archivos_wav}")
print(f" - Total de audios en buffer: {len(audio_buffers_list)}")

if audio_buffers_list:
    print("\nConcatenando y remuestreando a 22050 Hz...")
    babble_completo_16k = np.concatenate(audio_buffers_list)
    babble_audio_full = librosa.resample(babble_completo_16k, orig_sr=sr_original, target_sr=sr_objetivo)

    print("✅ Babble noise estructurado correctamente.")
    print(f"⏱️ Duración total del murmullo: {len(babble_audio_full)/sr_objetivo:.2f} segundos.")
else:
    print("\n❌ No hay audios válidos en la carpeta.")
    babble_audio_full = None

NameError: name 'babble_exact_path' is not defined

In [8]:

# ==================================================
# 3. FUNCIONES DE DATA AUGMENTATION
# ==================================================
def add_white_noise(audio, noise_level=0.005):
    noise = np.random.randn(len(audio))
    return audio + noise_level * noise

def add_pink_noise(audio, noise_level=0.01):
    white = np.random.randn(len(audio))
    fft_white = np.fft.rfft(white)
    frequencies = np.maximum(np.fft.rfftfreq(len(audio)), 1e-10)
    f_filter = 1.0 / np.sqrt(frequencies)
    f_filter /= np.max(f_filter)
    fft_pink = fft_white * f_filter
    pink = np.fft.irfft(fft_pink, n=len(audio))
    pink = pink / np.max(np.abs(pink))
    return audio + noise_level * pink

def add_babble_noise(audio, babble_audio, noise_level=0.03):
    if babble_audio is None:
        return audio

    if len(babble_audio) < len(audio):
        babble_audio = np.tile(babble_audio, int(np.ceil(len(audio) / len(babble_audio))))

    start_idx = random.randint(0, len(babble_audio) - len(audio))
    babble_chunk = babble_audio[start_idx : start_idx + len(audio)]
    babble_chunk = babble_chunk / (np.max(np.abs(babble_chunk)) + 1e-10)

    return audio + noise_level * babble_chunk

In [14]:
# ==========================================
# 4. TRANSFORMACIÓN Y DATA AUGMENTATION
# ==========================================
correct_audio_dir = None
dataset_root_path = kagglehub.dataset_download("mmoreaux/environmental-sound-classification-50")
metadata_path = os.path.join(dataset_root_path, 'esc50.csv')

if os.path.exists(metadata_path):
    df = pd.read_csv(metadata_path)
    print("Archivo de metadatos cargado correctamente.")
else:
    raise FileNotFoundError(f"No se encontró el archivo esc50.csv en la ruta: {metadata_path}")

# Mis clases
mis_clases = ['alarm', 'door_bell', 'cat', 'crying_baby', 'dog', 'shouting']
df_filtrado = df[df['category'].isin(mis_clases)].copy()


for root, dirs, files in os.walk(dataset_root_path):
    if any(f.endswith('.wav') for f in files):
        correct_audio_dir = root
        break

X_list = []
y_list = []

print(f"DataFrame listo: {len(df_filtrado)} audios base listos para ser multiplicados.")

for index, row in df_filtrado.iterrows():
    file_path = os.path.join(correct_audio_dir, row['filename'])
    categoria = row['category']

    try:
        y_audio, sr = librosa.load(file_path, sr=22050)
        # print(y_audio.shape)

        audios_a_procesar = [
            y_audio,                                     
            add_white_noise(y_audio),                    
            add_pink_noise(y_audio),                     
            add_babble_noise(y_audio, babble_audio_full) 
        ]

        for audio_version in audios_a_procesar:
            X_list.append(audio_version)
            y_list.append(categoria) 
        
    except:
        print("blabla")
X = np.array(X_list)
y = np.array(y_list)
print(X.shape)


Archivo de metadatos cargado correctamente.
DataFrame listo: 120 audios base listos para ser multiplicados.
(480, 110250)


In [23]:
y

array(['dog', 'dog', 'dog', 'dog', 'dog', 'dog', 'dog', 'dog',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby', 'dog',
       'dog', 'dog', 'dog', 'dog', 'dog', 'dog', 'dog', 'dog', 'dog',
       'dog', 'dog', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat',
       'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat',
       'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat',
       'cat', 'cat', 'cat', 'dog', 'dog', 'dog', 'dog', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'cat', 'cat', 'cat',
       'cat', 'dog', 'dog', 'dog', 'd

In [29]:
# ==========================================
# 5. ENTRENAMIENTO CON DETACH-ROCKET
# ==========================================
from sklearn.ensemble import RandomForestClassifier
le = LabelEncoder()
y_encoded = le.fit_transform(y)


X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
print("-" * 50)
print("RESUMEN DE AUDIOS UTILIZADOS:")
print(f"Total de muestras (con Data Augmentation): {len(X)}")
print(f" - Muestras para Entrenamiento: {len(X_train)}")
print(f" - Muestras para Prueba (Test): {len(X_test)}")
print("-" * 50)

inicio = time.time()

# Instanciar y ajustar Detach-ROCKET
detach_model = RocketClassifier(num_kernels=500)
detach_model.fit(X_train, y_train)

--------------------------------------------------
RESUMEN DE AUDIOS UTILIZADOS:
Total de muestras (con Data Augmentation): 480
 - Muestras para Entrenamiento: 384
 - Muestras para Prueba (Test): 96
--------------------------------------------------


/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/skbase/base/_base.py:1342: FutureWarning: tag 'handles-missing-data' will be removed in sktime version 1.0.0 and replaced by 'capability:missing_values', please use 'capability:missing_values' instead
  self._deprecate_tag_warn(collected_tags)
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


RocketClassifier(num_kernels=500)

In [ ]:

# Hacer predicciones sobre el conjunto de prueba
y_pred = detach_model.predict(X_test)

# Calcular la métrica con sklearn
accuracy = accuracy_score(y_test, y_pred)


/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/skbase/base/_base.py:1342: FutureWarning: tag 'handles-missing-data' will be removed in sktime version 1.0.0 and replaced by 'capability:missing_values', please use 'capability:missing_values' instead
  self._deprecate_tag_warn(collected_tags)
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [31]:
accuracy

1.0

In [ ]:



# Hacer predicciones sobre el conjunto de prueba
y_pred = detach_model.predict(X_test)

# Calcular la métrica con sklearn
accuracy = accuracy_score(y_test, y_pred)
fin = time.time()

print(f"Precisión (Test Accuracy): {accuracy * 100:.2f}%")
print(f"Tiempo total de ejecución: {fin - inicio:.2f} segundos")

# DetachRocket permite visualizar qué proporción de variables (features) retuvo:
if hasattr(detach_model, 'feature_proportion_'):
    print(f"Porcentaje de features retenidas tras el podado: {detach_model.feature_proportion_ * 100:.2f}%")

# ==========================================
# 6. AUDICIÓN DE VARIANTES
# ==========================================
if 'df_filtrado' in locals() and 'correct_audio_dir' in locals() and correct_audio_dir is not None:
    random_row = df_filtrado.sample(n=1).iloc[0]
    random_file_path = os.path.join(correct_audio_dir, random_row['filename'])
    random_label = random_row['category']

    print("=" * 50)
    print(f"AUDICIÓN DE VARIANTES: {random_label.upper()}")
    print(f"Archivo base: {random_row['filename']}")
    print("=" * 50)

    try:
        y_audio_test, sr_audio = librosa.load(random_file_path, sr=22050)
        
        print("\n1. Audio Original:")
        display(ipd.Audio(y_audio_test, rate=sr_audio))
        
        print("\n2. Variante: Ruido Blanco:")
        y_white = add_white_noise(y_audio_test)
        display(ipd.Audio(y_white, rate=sr_audio))
            
        print("\n3. Variante: Ruido Rosa:")
        y_pink = add_pink_noise(y_audio_test)
        display(ipd.Audio(y_pink, rate=sr_audio))
            
        if babble_audio_full is not None:
            print("\n4. Variante: Murmullo de fondo (Babble Noise):")
            y_babble = add_babble_noise(y_audio_test, babble_audio_full)
            display(ipd.Audio(y_babble, rate=sr_audio))
        else:
            print("\n[Aviso: No se generó Babble Noise válido]")
        
    except Exception as e:
        print(f"Error interno al intentar procesar los audios: {e}")

--------------------------------------------------
RESUMEN DE AUDIOS UTILIZADOS:
Total de muestras (con Data Augmentation): 480
 - Muestras para Entrenamiento: 384
 - Muestras para Prueba (Test): 96
--------------------------------------------------


ValueError: Found input variables with inconsistent numbers of samples: [1, 384]